In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

In [7]:
load_dotenv(".env")
CONNECTION_STRING = os.environ["BLOB_STORAGE_CONNECTION_STRING"]

In [8]:
def upload_file(local_path: Path, container: str, blob_name: str = None) -> None:
    """Upload a single file to a given container."""
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_name = blob_name or local_path.name

    blob_client = blob_service_client.get_blob_client(container=container, blob=blob_name)

    with open(local_path, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)

    print(f"Uploaded {local_path.name} -> {container}/{blob_name}")


def upload_folder(local_folder: Path, container: str) -> None:
    """Upload all files in a folder to a given container."""
    for file in local_folder.iterdir():
        if file.is_file():
            upload_file(file, container)

def list_all_containers():
    print("All containers:")
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    for container in blob_service_client.list_containers():
        print(f"   {container.name}")

def list_all_files_in_container(container):
    print(f"All files in {container}:")
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(container)
    for blob in container_client.list_blobs():
        print(f"   {blob.name}")

def delete_blob(container: str, blob_name: str) -> None:
    """Delete a single blob from a given container."""
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_client = blob_service_client.get_blob_client(container=container, blob=blob_name)

    try:
        blob_client.delete_blob()
        print(f"Deleted {container}/{blob_name}")
    except Exception as e:
        print(f"Failed to delete {container}/{blob_name}: {e}")



In [17]:
list_all_containers()
list_all_files_in_container("processed")

All containers:
   app-package-knmientsoedataingestion-8015333
   app-package-offshoreautomaticpreprocessingbl-2496851
   app-package-offshorewindapi-6209233
   azure-webjobs-hosts
   azure-webjobs-secrets
   knmi-entsoe-data-ingestion
   processed
   raw-entsoe
   raw-kaggle
   raw-knmi
   raw-offshore
   scm-releases
All files in processed:
   knmi_Hoek_van_Holland.csv
   knmi_Hoorn.csv
   knmi_Lauwersoog.csv
   knmi_Valkenburg.csv
   knmi_Vlieland.csv
   knmi_Vlissingen.csv
   knmi_Wilhelminadorp.csv


In [11]:
upload_file(local_path=Path("datasets_raw/knmi_dataset/De Kooy_235_2021-2030.txt"),container="raw-knmi")

Uploaded De Kooy_235_2021-2030.txt -> raw-knmi/De Kooy_235_2021-2030.txt


In [12]:
delete_blob("processed","knmi_De_Kooy.csv")

Deleted processed/knmi_De_Kooy.csv


In [14]:
upload_folder(Path("datasets_raw/offshore_dataset/"), "raw-offshore")

Uploaded Borssele_12.csv -> raw-offshore/Borssele_12.csv
Uploaded Borssele_34.csv -> raw-offshore/Borssele_34.csv
Uploaded Gemini.csv -> raw-offshore/Gemini.csv
Uploaded Hollandse_Kust_Noord.csv -> raw-offshore/Hollandse_Kust_Noord.csv
Uploaded Hollandse_Kust_Zuid.csv -> raw-offshore/Hollandse_Kust_Zuid.csv
Uploaded readme.txt -> raw-offshore/readme.txt


In [5]:
import os
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

import requests

load_dotenv(".env")
SAS_TOKEN = "?" + os.environ["BLOB_STORAGE_SAS_KEY"]

STORAGE_ACCOUNT = "windfarmstorageoffshore"
CONTAINER = "processed"

BASE_URL = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net/{CONTAINER}"
OUTPUT_DIR = "downloaded_processed"

def list_blobs() -> list[str]:
    """List all blob names in the container via the List Blobs REST API."""
    names = []
    marker = ""

    while True:
        url = f"{BASE_URL}{SAS_TOKEN}&restype=container&comp=list"
        if marker:
            url += f"&marker={marker}"

        response = requests.get(url)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        for blob in root.findall(".//Blob/Name"):
            names.append(blob.text)

        next_marker = root.findtext("NextMarker")
        if not next_marker:
            break
        marker = next_marker

    return names


def download_blob(blob_name: str, output_dir: str) -> None:
    """Download a single blob via plain HTTP GET."""
    url = f"{BASE_URL}/{blob_name}{SAS_TOKEN}"
    response = requests.get(url)
    response.raise_for_status()

    local_path = os.path.join(output_dir, blob_name)
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)

    with open(local_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded {blob_name} ({len(response.content)} bytes)")


def download_all_processed(output_dir: str = OUTPUT_DIR) -> None:
    os.makedirs(output_dir, exist_ok=True)

    blob_names = list_blobs()
    print(f"Found {len(blob_names)} files in '{CONTAINER}'")

    for name in blob_names:
        download_blob(name, output_dir)

    print(f"Done. Files saved to '{output_dir}/'")


if __name__ == "__main__":
    download_all_processed()

Found 7 files in 'processed'
Downloaded knmi_Hoek_van_Holland.csv (12034899 bytes)
Downloaded knmi_Hoorn.csv (11174900 bytes)
Downloaded knmi_Lauwersoog.csv (11072953 bytes)
Downloaded knmi_Valkenburg.csv (2617335 bytes)
Downloaded knmi_Vlieland.csv (11512918 bytes)
Downloaded knmi_Vlissingen.csv (11749371 bytes)
Downloaded knmi_Wilhelminadorp.csv (10850132 bytes)
Done. Files saved to 'downloaded_processed/'


In [2]:
load_dotenv(".env")
SAS_TOKEN = os.environ["BLOB_STORAGE_SAS_KEY"]